# Advanced Interpreter Test Suite
## Ground Truth Validation, Edge Cases, Confidence Thresholds, and Robustness Testing

This notebook comprehensively tests the Interpreter component against ground truth labels with multi-action support.

In [2]:
# ============================================================
# Cell 0: INITIALIZE EVERYTHING + DEFINE ALL FUNCTIONS
# ============================================================

import json
import os
import sys
import time
from pathlib import Path
from datetime import datetime
from groq import Groq
from dotenv import load_dotenv
import numpy as np

# ============ LOAD ENVIRONMENT ============
load_dotenv()
print("✓ Environment variables loaded")

# ============ SET UP FILE PATHS ============
# ============ SET UP FILE PATHS ============
# Handle both cases: notebooks in root OR in scripts/ subfolder
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'scripts':
    # If running from scripts/ folder, go up one level
    BASE_DIR = NOTEBOOK_DIR.parent
else:
    # If running from root folder
    BASE_DIR = NOTEBOOK_DIR
DATA_DIR = BASE_DIR / 'data'
KB_DIR = BASE_DIR / 'kb'
RESULTS_DIR = BASE_DIR / 'results'

print(f"Base directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"KB directory: {KB_DIR}")

# Create results directory if it doesn't exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ============ INITIALIZE GROQ CLIENT ============
api_key = os.getenv('GROQ_API_KEY')
if api_key:
    client = Groq(api_key=api_key)
    print("✓ Groq client initialized successfully")
else:
    print("⚠ WARNING: GROQ_API_KEY not found")

# ============ LOAD ALL DATA FILES ============
print("\n" + "="*60)
print("LOADING DATA FILES")
print("="*60)

# Load ground truth labels
with open(DATA_DIR / 'ground_truth_labels.json', 'r') as f:
    ground_truth_data = json.load(f)
    
    # Extract labels from the structure
    if isinstance(ground_truth_data, list):
        # File is directly a list of labels
        ground_truth_labels = ground_truth_data
    elif isinstance(ground_truth_data, dict) and 'labels' in ground_truth_data:
        # File has metadata + labels
        ground_truth_labels = ground_truth_data['labels']
    elif isinstance(ground_truth_data, dict) and 'label' in ground_truth_data:
        # Try singular 'label' as fallback
        ground_truth_labels = ground_truth_data['label']
    else:
        # If none of the above, try to extract any list from the dict
        for key, value in ground_truth_data.items():
            if isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
                ground_truth_labels = value
                print(f"Using '{key}' as labels")
                break
        else:
            print("✗ ERROR: Could not find labels in ground_truth_labels.json")
            ground_truth_labels = []
    
    print(f"✓ Loaded {len(ground_truth_labels)} ground truth labels")

# Load interpreter output (from test2)
with open(RESULTS_DIR / 'interpreter_output.json', 'r') as f:
    interpreter_data = json.load(f)
    interpreter_results = interpreter_data['results']
    print(f"✓ Loaded {len(interpreter_results)} interpreter results")

# Load dataset tables
with open(DATA_DIR / 'dataset_tables.json', 'r') as f:
    dataset_data = json.load(f)
    tables = dataset_data['tables']
    print(f"✓ Loaded {len(tables)} table schemas")

# Load knowledge base
with open(KB_DIR / 'knowledge_base.json', 'r') as f:
    kb_data = json.load(f)
    actions = kb_data['actions']
    print(f"✓ Loaded {len(actions)} actions from KB")

print(f"\n✓ All data loaded successfully. Ready for testing!")

# ============ INTERPRETER PROMPT ============
INTERPRETER_PROMPT = """Given a user query about vaccine adverse reaction analysis, extract the analytical intent.

Valid actions:
- PREDICT: Forecast future outcomes based on historical patterns
- COMPARE: Identify differences between groups or regions
- CORRELATE: Find relationships between variables
- SUMMARIZE: Describe current data or compute aggregates
- EXPLAIN: Understand why something happens
- FILTER: Select subset of data based on criteria
- JOIN: Combine data from multiple tables

For EACH query:
1. Identify the primary analytical intent (the main action)
2. Rate confidence (0-1) based on clarity of intent
3. Explain your reasoning briefly

Respond in JSON format:
{
  "intended_action": "ACTION_NAME",
  "confidence": 0.95,
  "reasoning": "Brief explanation"
}

If query is unclear or unrelated, respond with:
{
  "intended_action": "ERROR",
  "confidence": 0.0,
  "reasoning": "Explanation of why intent cannot be determined"
}"""

# ============ DEFINE extract_intent FUNCTION ============
def extract_intent(query, client):
    """Extract analytical intent from query using Groq API with rate limiting."""
    time.sleep(0.3)  # Rate limiting for Groq free tier (30 RPM)
    
    try:
        response = client.chat.completions.create(
            model="groq/compound",  # ← CHANGED THIS
            max_tokens=256,
            temperature=0.1,
            messages=[
                {"role": "user", "content": f"{INTERPRETER_PROMPT}\n\nQuery: {query}"}
            ]
        )
        response_text = response.choices[0].message.content
        result = json.loads(response_text)
        return result
    except json.JSONDecodeError:
        return {"intended_action": "ERROR", "confidence": 0.0, "reasoning": "Failed to parse response"}
    except Exception as e:
        return {"intended_action": "ERROR", "confidence": 0.0, "reasoning": str(e)}

# ============ DEFINE validate_single_query FUNCTION ============
def validate_single_query(query_str, interpreter_result, ground_truth_labels):
    """Validate a single query against ground truth.
    Returns: (is_correct, expected_actions, actual_action)
    """
    # Ensure ground_truth_labels is a list of dicts
    if not isinstance(ground_truth_labels, list):
        return False, "INVALID_GT", interpreter_result['intended_action']
    
    # Find ground truth for this query
    matching_gt = None
    for gt_item in ground_truth_labels:
        # Skip if not a dict
        if not isinstance(gt_item, dict):
            continue
        if gt_item.get('query') == query_str:
            matching_gt = gt_item
            break
    
    if not matching_gt:
        return False, "NOT_FOUND", interpreter_result['intended_action']
    
    expected_actions = matching_gt.get('ground_truth_actions', [])
    actual_action = interpreter_result['intended_action']
    
    # Check if actual action is in expected actions
    is_correct = actual_action in expected_actions
    
    return is_correct, expected_actions, actual_action

# ============ DEFINE compute_metrics FUNCTION ============
def compute_metrics(validation_results):
    """Compute accuracy metrics from validation results."""
    total = len(validation_results)
    correct = sum(1 for v in validation_results if v['is_correct'])
    
    single_action_qs = sum(1 for v in validation_results if len(v['expected_actions']) == 1)
    single_action_correct = sum(1 for v in validation_results if v['is_correct'] and len(v['expected_actions']) == 1)
    
    multi_action_qs = sum(1 for v in validation_results if len(v['expected_actions']) > 1)
    multi_action_correct = sum(1 for v in validation_results if v['is_correct'] and len(v['expected_actions']) > 1)
    
    return {
    'total_accuracy': correct / total if total > 0 else 0,
    'correct_count': correct,
    'total_count': total,
    'single_action_accuracy': single_action_correct / single_action_qs if single_action_qs > 0 else 0,
    'single_action_correct': single_action_correct,  # ADD THIS
    'single_action_count': single_action_qs,
    'multi_action_accuracy': multi_action_correct / multi_action_qs if multi_action_qs > 0 else 0,
    'multi_action_correct': multi_action_correct,    # ADD THIS
    'multi_action_count': multi_action_qs
}

print("\n✓ All functions defined and ready to use")

✓ Environment variables loaded
Base directory: /Users/enaso/Desktop/play ground/intention_grounding/_archive_2026-09-16
Data directory: /Users/enaso/Desktop/play ground/intention_grounding/_archive_2026-09-16/data
KB directory: /Users/enaso/Desktop/play ground/intention_grounding/_archive_2026-09-16/kb
✓ Groq client initialized successfully

LOADING DATA FILES


FileNotFoundError: [Errno 2] No such file or directory: '/Users/enaso/Desktop/play ground/intention_grounding/_archive_2026-09-16/data/ground_truth_labels.json'

In [2]:
# ============================================================
# Cell 1: Ground Truth Validation
# ============================================================

print("="*70)
print("GROUND TRUTH VALIDATION")
print("="*70)

validation_results = []

for i, interp_result in enumerate(interpreter_results, 1):
    query = interp_result['query']
    is_correct, expected, actual = validate_single_query(query, interp_result, ground_truth_labels)
    
    validation_results.append({
        'query_index': i,
        'query': query,
        'expected_actions': expected if isinstance(expected, list) else [expected],
        'actual_action': actual,
        'is_correct': is_correct,
        'confidence': interp_result['confidence']
    })
    
    status = "✓" if is_correct else "✗"
    print(f"\n{status} Q{i}: {query[:60]}...")
    print(f"    Expected: {expected if isinstance(expected, list) else [expected]}")
    print(f"    Got:      {actual} (confidence: {interp_result['confidence']:.2f})")

# Compute metrics
metrics = compute_metrics(validation_results)
print(f"\n" + "="*70)
print(f"VALIDATION METRICS")
print(f"="*70)
print(f"\nOverall Accuracy: {metrics['total_accuracy']:.2%} ({metrics['correct_count']}/{metrics['total_count']} correct)")
print(f"\nSingle-Action Queries:  {metrics['single_action_accuracy']:.2%} ({metrics['single_action_correct']}/{metrics['single_action_count']} correct)")
print(f"Multi-Action Queries:   {metrics['multi_action_accuracy']:.2%} ({metrics['multi_action_correct']}/{metrics['multi_action_count']} correct)")

# Save validation results
with open(RESULTS_DIR / 'validation_results.json', 'w') as f:
    json.dump({
        'metadata': {
            'timestamp': datetime.now().isoformat(),
            'total_queries': len(validation_results)
        },
        'results': validation_results,
        'metrics': metrics
    }, f, indent=2)

print(f"\n✓ Validation results saved to validation_results.json")

GROUND TRUTH VALIDATION

✓ Q1: Predict vaccine side effects in Europe for next month...
    Expected: ['PREDICT']
    Got:      PREDICT (confidence: 0.90)

✓ Q2: Compare side effect rates across vaccine brands...
    Expected: ['COMPARE']
    Got:      COMPARE (confidence: 0.95)

✓ Q3: Does age correlate with severe side effects?...
    Expected: ['CORRELATE']
    Got:      CORRELATE (confidence: 0.95)

✓ Q4: Summarize adverse events by region...
    Expected: ['SUMMARIZE']
    Got:      SUMMARIZE (confidence: 0.95)

✓ Q5: Find all severe adverse events reported in Q1 2026...
    Expected: ['FILTER']
    Got:      FILTER (confidence: 0.95)

✓ Q6: Link vaccine batches with adverse event reports to track bat...
    Expected: ['JOIN']
    Got:      JOIN (confidence: 0.95)

✓ Q7: Why do some regions report more side effects than others?...
    Expected: ['COMPARE', 'EXPLAIN']
    Got:      EXPLAIN (confidence: 0.95)

✓ Q8: Does vaccine batch production date correlate with event rate...
   

In [3]:
# ============================================================
# Cell 2: Edge Cases Test
# ============================================================

print("\n" + "="*70)
print("EDGE CASES TEST")
print("="*70)

edge_cases = [
    ("off_domain", "What's the weather in Paris?"),
    ("contradictory", "Predict what happened in the past using future data"),
    ("gibberish", "xyzabc qwerty vaccine side effects 123 !@#$"),
    ("factual_question", "How many people live in Europe?"),
    ("empty", "")
]

edge_results = []

for case_name, query in edge_cases:
    result = extract_intent(query, client)
    edge_results.append({
        'case_type': case_name,
        'query': query,
        'result': result
    })
    
    print(f"\n{case_name.upper()}:")
    print(f"  Query: {query[:60]}..." if len(query) > 60 else f"  Query: {query}")
    print(f"  Action: {result['intended_action']}")
    print(f"  Confidence: {result['confidence']:.2f}")
    print(f"  Reasoning: {result['reasoning'][:80]}..." if len(result['reasoning']) > 80 else f"  Reasoning: {result['reasoning']}")

error_count = sum(1 for e in edge_results if e['result']['intended_action'] == 'ERROR')
print(f"\n✓ Edge Cases: {error_count}/{len(edge_cases)} correctly returned ERROR")


EDGE CASES TEST

OFF_DOMAIN:
  Query: What's the weather in Paris?
  Action: ERROR
  Confidence: 0.00
  Reasoning: The query is unrelated to vaccine adverse reaction analysis and seems to be aski...

CONTRADICTORY:
  Query: Predict what happened in the past using future data
  Action: ERROR
  Confidence: 0.00
  Reasoning: Failed to parse response

GIBBERISH:
  Query: xyzabc qwerty vaccine side effects 123 !@#$
  Action: SUMMARIZE
  Confidence: 0.80
  Reasoning: The query contains the phrase 'vaccine side effects', indicating interest in dat...

FACTUAL_QUESTION:
  Query: How many people live in Europe?
  Action: SUMMARIZE
  Confidence: 0.90
  Reasoning: The query asks for a specific piece of information about the population of Europ...

EMPTY:
  Query: 
  Action: ERROR
  Confidence: 0.00
  Reasoning: The specific user query about vaccine adverse reaction analysis was not provided...

✓ Edge Cases: 3/5 correctly returned ERROR


In [5]:
# ============================================================
# Cell 3: Confidence Threshold Analysis
# ============================================================

print("\n" + "="*70)
print("CONFIDENCE THRESHOLD ANALYSIS")
print("="*70)

thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
threshold_analysis = []

for threshold in thresholds:
    # Count queries above threshold
    above_threshold = [r for r in validation_results if r['confidence'] >= threshold]
    correct_above = sum(1 for r in above_threshold if r['is_correct'])
    
    accuracy = correct_above / len(above_threshold) if len(above_threshold) > 0 else 0
    coverage = len(above_threshold) / len(validation_results)
    
    threshold_analysis.append({
        'threshold': threshold,
        'queries_above': len(above_threshold),
        'coverage': coverage,
        'accuracy': accuracy,
        'correct_count': correct_above
    })
    
    print(f"\nThreshold ≥ {threshold}:")
    print(f"  Coverage: {coverage:.1%} ({len(above_threshold)}/{len(validation_results)} queries)")
    print(f"  Accuracy: {accuracy:.1%} ({correct_above}/{len(above_threshold)} correct)")

print(f"\n✓ Threshold analysis complete")


CONFIDENCE THRESHOLD ANALYSIS

Threshold ≥ 0.5:
  Coverage: 100.0% (18/18 queries)
  Accuracy: 88.9% (16/18 correct)

Threshold ≥ 0.6:
  Coverage: 100.0% (18/18 queries)
  Accuracy: 88.9% (16/18 correct)

Threshold ≥ 0.7:
  Coverage: 100.0% (18/18 queries)
  Accuracy: 88.9% (16/18 correct)

Threshold ≥ 0.8:
  Coverage: 94.4% (17/18 queries)
  Accuracy: 94.1% (16/17 correct)

Threshold ≥ 0.9:
  Coverage: 83.3% (15/18 queries)
  Accuracy: 93.3% (14/15 correct)

Threshold ≥ 0.95:
  Coverage: 66.7% (12/18 queries)
  Accuracy: 91.7% (11/12 correct)

✓ Threshold analysis complete


In [6]:
# ============================================================
# Cell 4: Robustness Test (Synthetic - No New API Calls)
# ============================================================

print("\n" + "="*70)
print("ROBUSTNESS TEST: CONSISTENCY ANALYSIS")
print("="*70)

# Instead of making new API calls (which hit rate limits),
# test robustness by analyzing patterns in existing results

robustness_results = []

# Check if similar queries get similar actions
print("\nAnalyzing action consistency across query types:")

# Group queries by their ground truth actions
action_groups = {}
for i, interp_result in enumerate(interpreter_results):
    action = interp_result['intended_action']
    if action not in action_groups:
        action_groups[action] = []
    action_groups[action].append({
        'index': i,
        'query': interp_result['query'],
        'confidence': interp_result['confidence']
    })

# Analyze consistency
for action, queries in action_groups.items():
    if len(queries) < 2:
        continue
    
    print(f"\n{action} queries ({len(queries)} found):")
    
    # Check if interpreter is consistent with confidence
    confidences = [q['confidence'] for q in queries]
    avg_conf = sum(confidences) / len(confidences)
    conf_range = max(confidences) - min(confidences)
    
    print(f"  Average confidence: {avg_conf:.2f}")
    print(f"  Confidence range: {conf_range:.2f} (min: {min(confidences):.2f}, max: {max(confidences):.2f})")
    
    # Show first 2 examples
    print(f"  Examples:")
    for q in queries[:2]:
        print(f"    • Q{q['index']}: {q['query'][:60]}... (conf: {q['confidence']:.2f})")
    
    robustness_results.append({
        'action': action,
        'count': len(queries),
        'avg_confidence': avg_conf,
        'consistency_score': 1.0 - (conf_range / 2.0)  # Higher = more consistent
    })

print(f"\n" + "="*70)
print("CONSISTENCY SUMMARY")
print("="*70)

if robustness_results:
    avg_consistency = sum(r['consistency_score'] for r in robustness_results) / len(robustness_results)
    print(f"\nOverall Consistency Score: {avg_consistency:.1%}")
    print(f"(Based on confidence variance across action types)")
    
    print(f"\nAction-wise consistency:")
    for r in robustness_results:
        print(f"  {r['action']:12s}: {r['consistency_score']:.1%} ({r['count']} queries, avg conf: {r['avg_confidence']:.2f})")
else:
    print("No action groups found for analysis")

print(f"\n✓ Robustness analysis complete (no new API calls)")


ROBUSTNESS TEST: CONSISTENCY ANALYSIS

Analyzing action consistency across query types:

PREDICT queries (2 found):
  Average confidence: 0.93
  Confidence range: 0.05 (min: 0.90, max: 0.95)
  Examples:
    • Q0: Predict vaccine side effects in Europe for next month... (conf: 0.90)
    • Q9: Forecast adverse event trends by age group over the next 12 ... (conf: 0.95)

COMPARE queries (3 found):
  Average confidence: 0.95
  Confidence range: 0.00 (min: 0.95, max: 0.95)
  Examples:
    • Q1: Compare side effect rates across vaccine brands... (conf: 0.95)
    • Q8: Compare trial efficacy outcomes for Moderna vs Pfizer across... (conf: 0.95)

CORRELATE queries (3 found):
  Average confidence: 0.95
  Confidence range: 0.00 (min: 0.95, max: 0.95)
  Examples:
    • Q2: Does age correlate with severe side effects?... (conf: 0.95)
    • Q7: Does vaccine batch production date correlate with event rate... (conf: 0.95)

SUMMARIZE queries (4 found):
  Average confidence: 0.90
  Confidence range: 0

In [7]:
# ============================================================
# Cell 5: Summary Report
# ============================================================

print("\n" + "="*70)
print("ADVANCED TEST SUMMARY")
print("="*70)

# Compile comprehensive summary
summary_report = {
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'test_suite': 'advanced_interpreter_test_v3',
        'components': ['ground_truth_validation', 'edge_cases', 'confidence_thresholds', 'robustness']
    },
    'validation': {
        'metrics': metrics,
        'total_queries_tested': len(validation_results),
        'summary': f"Overall accuracy: {metrics['total_accuracy']:.1%} on {metrics['total_count']} queries"
    },
    'edge_cases': {
        'total_cases': len(edge_results),
        'error_returns': sum(1 for e in edge_results if e['result']['intended_action'] == 'ERROR'),
        'summary': f"Correctly returned ERROR for {sum(1 for e in edge_results if e['result']['intended_action'] == 'ERROR')}/{len(edge_results)} edge cases"
    },
    'confidence_thresholds': threshold_analysis,
    'robustness': {
        'total_tests': len(robustness_results),
        'avg_consistency': np.mean([r['consistency_score'] for r in robustness_results]) if robustness_results else 0,
        'details': robustness_results
    },
    'conclusion': {
        'interpreter_status': 'VALIDATED' if metrics['total_accuracy'] > 0.8 else 'NEEDS_REVIEW',
        'confidence_trend': 'Accurate at high confidence (≥0.9)',
        'edge_case_handling': 'Robust' if error_count >= 3 else 'Needs improvement',
        'consistency': 'Good' if (robustness_results and np.mean([r['consistency_score'] for r in robustness_results]) > 0.7) else 'Moderate'
    }
}

# Save summary report
with open(RESULTS_DIR / 'advanced_test_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print(f"\nValidation: {metrics['total_accuracy']:.1%} accuracy")
print(f"Edge Cases: {sum(1 for e in edge_results if e['result']['intended_action'] == 'ERROR')}/{len(edge_results)} error returns")
print(f"Robustness: {np.mean([r['consistency_score'] for r in robustness_results]):.1%} avg consistency" if robustness_results else "Robustness: N/A")
print(f"\nStatus: {summary_report['conclusion']['interpreter_status']}")
print(f"\n✓ Summary report saved to advanced_test_summary.json")


ADVANCED TEST SUMMARY

Validation: 88.9% accuracy
Edge Cases: 3/5 error returns
Robustness: 95.0% avg consistency

Status: VALIDATED

✓ Summary report saved to advanced_test_summary.json
